# Calculate shared colour limits for maps

Define data limits so that maps can be drawn on the same colour scales for all datasets.

In [1]:
import os
import pandas as pd

In [2]:
dir_output = 'output'
# Scenario results:
file_results_comparison = 'results_full_comparison_taunworc_base'
file_geo = 'results_geo_outcomes'
file_results = 'results_full'
# Output:
file_all_vlim = 'vlims_full.csv'
file_vlim_summary = 'vlims_summary.csv'
file_all_vlim_taunworc = 'vlims_full_taunworc.csv'
file_vlim_summary_taunworc = 'vlims_summary_taunworc.csv'

In [3]:
scenarios = ['base', 'taunworc']

## Set up data to gather



In [4]:
cols_vlims = [
    'nearest_ivt_time_no_bias',
    'drip_ship_mt_time',
    'drip_ship_mt_travel_time',
    'drip_ship_mt_travel_time_with_delay',
    'drip_ship_mt_travel_time_with_delay_base',
    'drip_ship_mrs_0-2_improvement_over_no_treatment',
    'lvo_drip_ship_ivt_mt_mrs_0-2_improvement_over_no_treatment',
    'change_in_drip_ship_mt_travel_time',
    'change_in_drip_ship_mt_time',
    'change_in_drip_ship_mt_travel_time_with_delay',
    'change_in_lvo_drip_ship_ivt_mt_mrs_0-2',
]

In [5]:
df_vlims = pd.DataFrame(index=cols_vlims, dtype=float)

## Gather file names

In [6]:
list_of_files = []

for s in '', '_bias10min':
    s_bias = 'nobias' if len(s) == 0 else s
    
    f1 = file_results_comparison + s + '.csv'
    list_of_files.append(f1)

    for f in [file_geo, file_results]:
        for scen in ['_base', '_taunworc']:
            f2 = f + scen + s + '.csv'
            list_of_files.append(f2)

In [7]:
list_of_files

['results_full_comparison_taunworc_base.csv',
 'results_geo_outcomes_base.csv',
 'results_geo_outcomes_taunworc.csv',
 'results_full_base.csv',
 'results_full_taunworc.csv',
 'results_full_comparison_taunworc_base_bias10min.csv',
 'results_geo_outcomes_base_bias10min.csv',
 'results_geo_outcomes_taunworc_bias10min.csv',
 'results_full_base_bias10min.csv',
 'results_full_taunworc_bias10min.csv']

In [8]:
list_of_files_taunworc = [f for f in list_of_files if 'comparison' in f]

list_of_files_taunworc

['results_full_comparison_taunworc_base.csv',
 'results_full_comparison_taunworc_base_bias10min.csv']

## Find vlims

Load in each dataset and gather data limits:

In [9]:
def gather_vlims(list_of_files, dir_output, cols_vlims, limit_to_change=False):
    dfs_to_concat = []
    
    for f in list_of_files:
        df = pd.read_csv(os.path.join(dir_output, f))
        if limit_to_change:
            # Limit to LSOA that have changed with the opening
            # of a Worcester or Taunton unit:
            df = df[df['changed']]
        else:
            pass
        # Gather relevant columns:
        cols = [c for c in df.columns if
                any([c.startswith(col_vlim) for col_vlim in cols_vlims])]
        # Record which target column each matches:
        cols_match = []
        for c in cols:
            cols_start = [v for v in cols_vlims if c.startswith(v)]
            # Move longest column to the front:
            cols_start = sorted(cols_start, key=len)[::-1]
            if len(cols_start) > 0:
                cols_match.append(cols_start[0])
            else:
                print('Problem:', c)
        if len(cols) != len(cols_match):
            print('Check matches')
            print(cols)
            print(cols_match)
        # Find minimum and maximum values:
        df = df[cols].describe().loc[['min', 'max']]
        df.loc['file'] = f
        df = df.transpose()
        df['col_vlim'] = cols_match
        df.index.name = 'col'
        df = df.reset_index()
        dfs_to_concat.append(df)
    
    df_limits = pd.concat(dfs_to_concat, axis='rows', ignore_index=True)
    df_limits = df_limits.sort_values('col_vlim')
    
    # Gather min and max values more simply:
    df_vlim = pd.DataFrame(columns=['min', 'max'])
    df_vlim.index.name = 'column'
    
    for col_vlim, df in df_limits.groupby('col_vlim'):
        vmin = df['min'].min()
        vmax = df['max'].max()
        df_vlim.loc[col_vlim] = [vmin, vmax]
    return df_limits, df_vlim

In [10]:
df_limits, df_vlim = gather_vlims(list_of_files, dir_output, cols_vlims)
df_limits_taunworc, df_vlim_taunworc = gather_vlims(
    list_of_files_taunworc, dir_output, cols_vlims, limit_to_change=True)

/tmp/ipykernel_48200/523260651.py:5: DtypeWarning: Columns (5: nearest_ivt_unit_multiple_same_time_and_distance_options, 13: nearest_mt_unit_multiple_same_time_and_distance_options, 18: transfer_unit_multiple_same_time_and_distance_options) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(os.path.join(dir_output, f))
/tmp/ipykernel_48200/523260651.py:5: DtypeWarning: Columns (5: nearest_ivt_unit_multiple_same_time_and_distance_options, 13: nearest_mt_unit_multiple_same_time_and_distance_options, 18: transfer_unit_multiple_same_time_and_distance_options) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(os.path.join(dir_output, f))
/tmp/ipykernel_48200/523260651.py:5: DtypeWarning: Columns (5: nearest_ivt_unit_multiple_same_time_and_distance_options, 13: nearest_mt_unit_multiple_same_time_and_distance_options, 18: transfer_unit_multiple_same_time_and_distance_options) have mixed types. Specify dtype o

In [11]:
display(df_limits.head())

,col,min,max,file,col_vlim
6,change_in_drip_ship_mt_time,-117.0,0.0,results_full_comparison_taunworc_base.csv,change_in_drip_ship_mt_time
33,change_in_drip_ship_mt_time,-136.0,0.0,results_full_comparison_taunworc_base_bias10mi...,change_in_drip_ship_mt_time
7,change_in_drip_ship_mt_travel_time,-57.0,0.0,results_full_comparison_taunworc_base.csv,change_in_drip_ship_mt_travel_time
34,change_in_drip_ship_mt_travel_time,-76.0,0.0,results_full_comparison_taunworc_base_bias10mi...,change_in_drip_ship_mt_travel_time
35,change_in_drip_ship_mt_travel_time_with_delay,-136.0,0.0,results_full_comparison_taunworc_base_bias10mi...,change_in_drip_ship_mt_travel_time_with_delay


In [12]:
display(df_vlim)

,min,max
column,,
change_in_drip_ship_mt_time,-136.000,0.000
change_in_drip_ship_mt_travel_time,-76.000,0.000
change_in_drip_ship_mt_travel_time_with_delay,-136.000,0.000
change_in_lvo_drip_ship_ivt_mt_mrs_0-2,0.000,0.086
drip_ship_mrs_0-2_improvement_over_no_treatment,0.082,0.139
drip_ship_mt_time,170.000,362.000
drip_ship_mt_travel_time,0.000,132.000
drip_ship_mt_travel_time_with_delay,0.000,192.000
drip_ship_mt_travel_time_with_delay_base,0.000,192.000


In [13]:
display(df_vlim_taunworc)

,min,max
column,,
change_in_drip_ship_mt_time,-136.0,0.000
change_in_drip_ship_mt_travel_time,-76.0,0.000
change_in_drip_ship_mt_travel_time_with_delay,-136.0,0.000
change_in_lvo_drip_ship_ivt_mt_mrs_0-2,0.0,0.086
drip_ship_mt_time,170.0,354.000
drip_ship_mt_travel_time,0.0,124.000
drip_ship_mt_travel_time_with_delay,0.0,160.000
drip_ship_mt_travel_time_with_delay_base,19.0,184.000


## Save results

In [14]:
p = os.path.join(dir_output, file_all_vlim)
df_limits.to_csv(p, index=False)

p = os.path.join(dir_output, file_vlim_summary)
df_vlim.to_csv(p, index=True)

p = os.path.join(dir_output, file_all_vlim_taunworc)
df_limits_taunworc.to_csv(p, index=False)

p = os.path.join(dir_output, file_vlim_summary_taunworc)
df_vlim_taunworc.to_csv(p, index=True)

## Check Taunton and Worcester

In [15]:
f = list_of_files_taunworc[1]
print(f)
df = pd.read_csv(os.path.join(dir_output, f))

results_full_comparison_taunworc_base_bias10min.csv


In [16]:
c = 'drip_ship_mt_travel_time_with_delay'
cols = [col for col in df.columns if col.startswith(c)]

df[df['changed']][cols].sort_values(c+'_taunworc')

,drip_ship_mt_travel_time_with_delay_base,drip_ship_mt_travel_time_with_delay_taunworc
27892,114,0
32309,90,2
32319,90,2
32311,90,2
27887,116,2
...,...,...
6944,181,157
6940,181,157
6939,182,158
6941,182,158
